[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishansurdi/SDK-Sovereign/blob/master/notebooks/00_all_in_one_pipeline.ipynb)

# SDK-Sovereign All-in-One Pipeline

Single notebook for smoke test, rollout collection, Lead GRPO, Auditor GRPO, evaluation, and plots.

## Required Colab Secrets
- HF_TOKEN
- WANDB_API_KEY (optional but recommended)

## Runtime
- GPU: T4
- Run cells top to bottom

In [ ]:
# Cell 1 - Install dependencies (pinned compatible stack)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U "trl==0.24.0" "datasets==3.6.0" "accelerate==1.6.0" "pydantic==2.10.6" peft bitsandbytes
!pip install -q "openenv-core[core] @ git+https://github.com/meta-pytorch/OpenEnv@v0.2.3"
!pip install -q wandb huggingface_hub matplotlib numpy
print('OK: dependencies installed')

In [ ]:
# Cell 2 - Clone repo, configure, and authenticate
import os
import sys
import json
import warnings
from pathlib import Path

from google.colab import userdata
from huggingface_hub import login

warnings.filterwarnings('ignore', message='Both `max_new_tokens`')
warnings.filterwarnings('ignore', category=FutureWarning, module='transformers.modeling_attn_mask_utils')

GH_REPO = 'ishansurdi/SDK-Sovereign'
HF_USER = 'ishansurdi'
ENV_URL = 'https://ishansurdi-sdk-sovereign.hf.space'
WANDB_PROJECT = 'sdk-sovereign'
WANDB_ENTITY = 'OpenEnvR2'

N_ROLLOUT_EPISODES = 80
N_EVAL_EPISODES = 30

RUN_SMOKE = True
RUN_ROLLOUTS = True
RUN_TRAIN_LEAD = True
RUN_TRAIN_AUDITOR = True
RUN_EVAL = True
PUSH_ADAPTERS = True

FORCE_ROLLOUT_REBUILD = False
FORCE_TRAIN_REBUILD = False

!git clone https://github.com/{GH_REPO}.git sdk_sovereign_pkg 2>/dev/null || (cd sdk_sovereign_pkg && git pull)
!pip install -q -e sdk_sovereign_pkg/
sys.path.insert(0, 'sdk_sovereign_pkg')

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

wandb_key = userdata.get('WANDB_API_KEY')
if wandb_key:
    os.environ['WANDB_API_KEY'] = wandb_key
    import wandb
    wandb.login(key=wandb_key)

Path('plots').mkdir(exist_ok=True)
print('OK: repo ready and auth complete')

In [ ]:
# Cell 3 - Load model/agents and utility helpers
import subprocess
from collections import defaultdict

from server.llm_agents import load_model_with_two_adapters, make_agent_pair
from client import SDKSovereignEnv
from models import SDKAction, SDKObservation

def load_grpo_symbols():
    try:
        from trl.trainer.grpo_trainer import GRPOTrainer, GRPOConfig
        import trl
        print(f'Using trl=={trl.__version__}')
        return GRPOTrainer, GRPOConfig
    except Exception:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', '-U',
            'trl==0.24.0', 'datasets==3.6.0', 'accelerate==1.6.0', 'pydantic==2.10.6'
        ])
        for key in list(sys.modules.keys()):
            if key == 'trl' or key.startswith('trl.'):
                del sys.modules[key]
        from trl.trainer.grpo_trainer import GRPOTrainer, GRPOConfig
        import trl
        print(f'Using trl=={trl.__version__}')
        return GRPOTrainer, GRPOConfig

def init_wandb_run(run_name):
    import wandb
    kwargs = {'project': WANDB_PROJECT, 'name': run_name}
    if WANDB_ENTITY:
        kwargs['entity'] = WANDB_ENTITY
    try:
        return wandb.init(**kwargs)
    except Exception as exc:
        print(f'W&B init failed with entity={WANDB_ENTITY!r}: {exc}')
        print('Retrying with default logged-in entity...')
        kwargs.pop('entity', None)
        return wandb.init(**kwargs)

def load_jsonl(path):
    p = Path(path)
    if not p.exists():
        return []
    rows = [ln for ln in p.read_text().splitlines() if ln.strip()]
    return [json.loads(r) for r in rows]

def save_jsonl(path, rows):
    Path(path).write_text('\n'.join(json.dumps(r) for r in rows))

model, tokenizer = load_model_with_two_adapters()
agents = make_agent_pair(model, tokenizer)
print('OK: model and agents loaded')

In [ ]:
# Cell 4 - Smoke test + adapter swap check
if RUN_SMOKE:
    print(f'Connecting to: {ENV_URL}')
    with SDKSovereignEnv(base_url=ENV_URL).sync() as env:
        obs = env.reset()
        print(f'Episode started | role={obs.current_role}')
        for turn in range(7):
            agent = agents[obs.current_role]
            action = agent(obs)
            print(f'Turn {turn:2d} | {obs.current_role:7s} | {action.action_type}')
            obs = env.step(action)
            if obs.done:
                print(f'Done! reward={obs.reward:.2f} | breakdown={obs.reward_breakdown}')
                break

    dummy_aud = SDKObservation(
        current_role='auditor', turn_index=0, max_turns=7, error_log='test',
        conversation_history=[], visible_allowlist=['razorpay'], visible_codebase=None,
        visible_filename=None, current_proposal=None, approved_replacement=None,
        done=False, reward=0.0, reward_breakdown={},
    )
    agents['auditor'](dummy_aud)
    assert model.active_adapter == 'auditor_adapter'

    dummy_lead = SDKObservation(
        current_role='lead', turn_index=1, max_turns=7, error_log='test',
        conversation_history=[], visible_allowlist=None, visible_codebase='import stripe',
        visible_filename='payment.py', current_proposal=None, approved_replacement=None,
        done=False, reward=0.0, reward_breakdown={},
    )
    agents['lead'](dummy_lead)
    assert model.active_adapter == 'lead_adapter'
    print('OK: smoke and adapter swap checks passed')
else:
    print('Skipped smoke by config')

In [ ]:
# Cell 5 - Resume-safe rollout collection
if RUN_ROLLOUTS:
    rollout_lead_path = Path('rollout_lead.jsonl')
    rollout_auditor_path = Path('rollout_auditor.jsonl')
    rollout_data = {
        'lead': load_jsonl(rollout_lead_path),
        'auditor': load_jsonl(rollout_auditor_path),
    }

    if rollout_data['lead'] and rollout_data['auditor'] and not FORCE_ROLLOUT_REBUILD:
        lead_has_ep = all('episode' in r for r in rollout_data['lead'])
        aud_has_ep = all('episode' in r for r in rollout_data['auditor'])
        if lead_has_ep and aud_has_ep:
            done_lead = max((r['episode'] for r in rollout_data['lead']), default=-1) + 1
            done_aud = max((r['episode'] for r in rollout_data['auditor']), default=-1) + 1
            start_ep = min(done_lead, done_aud)
        else:
            start_ep = N_ROLLOUT_EPISODES
            print('Reusing existing rollout files (legacy format)')
    else:
        if FORCE_ROLLOUT_REBUILD:
            rollout_data = {'lead': [], 'auditor': []}
        start_ep = 0

    for ep in range(start_ep, N_ROLLOUT_EPISODES):
        with SDKSovereignEnv(base_url=ENV_URL).sync() as env:
            obs = env.reset()
            per_role = []
            while not obs.done and obs.turn_index < 7:
                agent = agents[obs.current_role]
                agent.model.set_adapter(agent.adapter_name)
                prompt = agent._build_prompt(obs)
                completion = agent._generate(prompt)
                action = agent._parse_action(completion)
                new_obs = env.step(action)
                per_role.append({
                    'episode': ep,
                    'role': obs.current_role,
                    'prompt': prompt,
                    'reward': float(new_obs.reward),
                })
                obs = new_obs

        for r in per_role:
            rollout_data[r['role']].append({
                'episode': r['episode'],
                'prompt': r['prompt'],
                'reward': r['reward'],
            })

        save_jsonl(rollout_lead_path, rollout_data['lead'])
        save_jsonl(rollout_auditor_path, rollout_data['auditor'])
        if ep % 5 == 0:
            print(f'rollout {ep}/{N_ROLLOUT_EPISODES}')

    print(f"Lead samples: {len(rollout_data['lead'])}")
    print(f"Auditor samples: {len(rollout_data['auditor'])}")
else:
    rollout_data = {
        'lead': load_jsonl('rollout_lead.jsonl'),
        'auditor': load_jsonl('rollout_auditor.jsonl'),
    }
    print('Skipped rollouts by config')

In [ ]:
# Cell 6 - Train Lead adapter (resume-safe)
if RUN_TRAIN_LEAD:
    GRPOTrainer, GRPOConfig = load_grpo_symbols()
    from datasets import Dataset
    import wandb

    init_wandb_run('lead-grpo-round1')

    lead_prompts = [r['prompt'] for r in rollout_data['lead']]
    ds_lead = Dataset.from_dict({'prompt': lead_prompts})

    def lead_reward_fn(completions, **kwargs):
        rewards = []
        for completion in completions:
            action = agents['lead']._parse_action(completion)
            if action.action_type == 'submit_patch':
                with SDKSovereignEnv(base_url=ENV_URL).sync() as env:
                    env.reset()
                    env.step(SDKAction(role='auditor', action_type='pass'))
                    env.step(SDKAction(role='lead', action_type='propose_replacement', proposed_sdk='razorpay'))
                    env.step(SDKAction(role='auditor', action_type='approve'))
                    final = env.step(action)
                    rewards.append(float(final.reward))
            elif action.action_type == 'propose_replacement':
                rewards.append(1.0)
            elif action.action_type == 'request_hint':
                rewards.append(0.3)
            else:
                rewards.append(-0.5)
        return rewards

    config = GRPOConfig(
        output_dir='checkpoints/lead',
        num_generations=4,
        max_completion_length=512,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=5e-6,
        num_train_epochs=1,
        logging_steps=2,
        save_steps=50,
        report_to='wandb',
    )

    lead_artifact_dir = Path('artifacts/lead_adapter')
    already_trained = (lead_artifact_dir / 'adapter_config.json').exists()

    if already_trained and not FORCE_TRAIN_REBUILD:
        print('Lead adapter already exists, skipping train')
    else:
        ckpt_dir = Path('checkpoints/lead')
        checkpoints = sorted(
            [p for p in ckpt_dir.glob('checkpoint-*') if p.name.split('-')[-1].isdigit()],
            key=lambda p: int(p.name.split('-')[-1]),
        )
        resume_from = str(checkpoints[-1]) if checkpoints else None
        if resume_from:
            print(f'Resuming from checkpoint: {resume_from}')

        trainer = GRPOTrainer(
            model=agents['lead'].model,
            processing_class=tokenizer,
            reward_funcs=[lead_reward_fn],
            args=config,
            train_dataset=ds_lead,
            peft_config=None,
            formatting_func=lambda ex: ex['prompt'],
        )
        trainer.train(resume_from_checkpoint=resume_from)
        agents['lead'].model.save_pretrained('artifacts/lead_adapter')
        tokenizer.save_pretrained('artifacts/lead_adapter')
        print('OK: lead adapter trained')

    if PUSH_ADAPTERS:
        from huggingface_hub import HfApi
        api = HfApi()
        model.save_pretrained('checkpoints/lead_adapter_v1', selected_adapters=['lead_adapter'])
        api.create_repo(repo_id=f'{HF_USER}/sdk-sovereign-lead-adapter', repo_type='model', exist_ok=True)
        api.upload_folder(
            folder_path='checkpoints/lead_adapter_v1',
            repo_id=f'{HF_USER}/sdk-sovereign-lead-adapter',
            repo_type='model',
            commit_message='Lead LoRA adapter v1 (GRPO round 1)',
        )
        print(f'OK: pushed {HF_USER}/sdk-sovereign-lead-adapter')
else:
    print('Skipped lead training by config')

In [ ]:
# Cell 7 - Train Auditor adapter (resume-safe)
if RUN_TRAIN_AUDITOR:
    GRPOTrainer, GRPOConfig = load_grpo_symbols()
    from datasets import Dataset
    from server.environment import SDKSovereignEnvironment
    import wandb

    init_wandb_run('auditor-grpo-round1')

    ds_auditor = Dataset.from_dict({'prompt': [x['prompt'] for x in rollout_data['auditor']]})

    def auditor_reward_fn(completions, **kwargs):
        rewards = []
        allowlist = set(SDKSovereignEnvironment().allowlist)
        for completion in completions:
            action = agents['auditor']._parse_action(completion)
            if action.action_type == 'approve':
                rewards.append(1.0)
            elif action.action_type == 'reject':
                rewards.append(0.5 if (action.rejection_reason or '').strip() else 0.1)
            elif action.action_type == 'give_hint':
                text = (action.hint_response or '').lower()
                hit = any(sdk.lower() in text for sdk in allowlist)
                rewards.append(0.8 if hit else -0.2)
            elif action.action_type == 'pass':
                rewards.append(-0.5)
            else:
                rewards.append(-0.2)
        return rewards

    config = GRPOConfig(
        output_dir='checkpoints/auditor',
        num_generations=4,
        max_completion_length=256,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=5e-6,
        num_train_epochs=1,
        logging_steps=2,
        report_to='wandb',
    )

    model.set_adapter('auditor_adapter')
    for n, p in model.named_parameters():
        p.requires_grad = ('auditor_adapter' in n)
    for n, p in model.named_parameters():
        if 'lead_adapter' in n:
            p.requires_grad = False

    trainer = GRPOTrainer(
        model=model,
        reward_funcs=auditor_reward_fn,
        args=config,
        train_dataset=ds_auditor.select(range(min(60, len(ds_auditor)))),
        tokenizer=tokenizer,
    )
    trainer.train()
    wandb.finish()
    print('OK: auditor adapter trained')

    if PUSH_ADAPTERS:
        from huggingface_hub import HfApi
        api = HfApi()
        model.save_pretrained('checkpoints/auditor_adapter_v1', selected_adapters=['auditor_adapter'])
        api.create_repo(repo_id=f'{HF_USER}/sdk-sovereign-auditor-adapter', repo_type='model', exist_ok=True)
        api.upload_folder(
            folder_path='checkpoints/auditor_adapter_v1',
            repo_id=f'{HF_USER}/sdk-sovereign-auditor-adapter',
            repo_type='model',
            commit_message='Auditor LoRA adapter v1 (GRPO round 1)',
        )
        print(f'OK: pushed {HF_USER}/sdk-sovereign-auditor-adapter')
else:
    print('Skipped auditor training by config')

In [ ]:
# Cell 8 - Evaluate baseline vs trained adapters
if RUN_EVAL:
    import server.llm_agents as la

    baseline_model, baseline_tok = la.load_model_with_two_adapters()
    baseline_agents = la.make_agent_pair(baseline_model, baseline_tok)

    trained_model, trained_tok = la.load_model_with_two_adapters()
    trained_model.load_adapter(f'{HF_USER}/sdk-sovereign-lead-adapter', adapter_name='lead_adapter_trained')
    trained_model.load_adapter(f'{HF_USER}/sdk-sovereign-auditor-adapter', adapter_name='auditor_adapter_trained')
    trained_agents = la.make_agent_pair(trained_model, trained_tok)
    trained_agents['lead'].adapter_name = 'lead_adapter_trained'
    trained_agents['auditor'].adapter_name = 'auditor_adapter_trained'

    def run_eval(eval_agents, label, n=N_EVAL_EPISODES):
        results = []
        for ep in range(n):
            with SDKSovereignEnv(base_url=ENV_URL).sync() as env:
                obs = env.reset()
                total = 0.0
                transcript = []
                while not obs.done and obs.turn_index < 7:
                    action = eval_agents[obs.current_role](obs)
                    transcript.append({
                        'turn': obs.turn_index,
                        'role': obs.current_role,
                        'action_type': action.action_type,
                    })
                    obs = env.step(action)
                    total += obs.reward
                state = env.state()
                tr = state.test_results or {}
                results.append({
                    'total_reward': total,
                    'tests_passed': sum(tr.values()),
                    'tests_total': len(tr) or 3,
                    'success': bool(tr and all(tr.values())),
                    'turns': state.step_count,
                    'repo': state.repo_id,
                    'terminated': state.terminated_reason,
                    'transcript': transcript,
                })
            if ep % 5 == 0:
                print(f'[{label}] ep {ep}/{n}')
        return results

    baseline_results = run_eval(baseline_agents, 'baseline')
    trained_results = run_eval(trained_agents, 'trained')
    Path('eval_results.json').write_text(json.dumps({'baseline': baseline_results, 'trained': trained_results}, indent=2))
    print('OK: eval complete')
else:
    baseline_results = []
    trained_results = []
    print('Skipped eval by config')

In [ ]:
# Cell 9 - Generate 6 plots
if RUN_EVAL and baseline_results and trained_results:
    import os
    import numpy as np
    import matplotlib.pyplot as plt
    import statistics

    b_rate = sum(r['success'] for r in baseline_results) / len(baseline_results)
    t_rate = sum(r['success'] for r in trained_results) / len(trained_results)

    plt.figure(figsize=(6, 4))
    plt.bar(['Baseline', 'Trained'], [b_rate, t_rate], color=['#bbbbbb', '#1f77b4'])
    plt.ylabel('Pass rate')
    plt.title('Pass rate, n=30 each')
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig('plots/pass_rate_baseline_vs_trained.png', dpi=150, bbox_inches='tight')
    plt.close()

    b_r = statistics.mean(r['total_reward'] for r in baseline_results)
    t_r = statistics.mean(r['total_reward'] for r in trained_results)
    plt.figure(figsize=(6, 4))
    plt.bar(['Baseline', 'Trained'], [b_r, t_r], color=['#bbbbbb', '#1f77b4'])
    plt.ylabel('Mean episode reward')
    plt.axhline(0, color='k', lw=0.5)
    plt.title('Mean total reward per episode')
    plt.tight_layout()
    plt.savefig('plots/mean_reward.png', dpi=150, bbox_inches='tight')
    plt.close()

    x = np.arange(2)
    plt.figure(figsize=(6, 4))
    plt.plot(x, [b_r, t_r], marker='o')
    plt.xticks(x, ['Baseline', 'Trained'])
    plt.ylabel('Reward')
    plt.title('Reward curve summary (Lead)')
    plt.tight_layout()
    plt.savefig('plots/reward_curve_lead.png', dpi=150, bbox_inches='tight')
    plt.close()

    plt.figure(figsize=(6, 4))
    plt.plot(x, [b_rate, t_rate], marker='o')
    plt.xticks(x, ['Baseline', 'Trained'])
    plt.ylabel('Pass rate')
    plt.title('Reward curve summary (Auditor)')
    plt.tight_layout()
    plt.savefig('plots/reward_curve_auditor.png', dpi=150, bbox_inches='tight')
    plt.close()

    b_per = defaultdict(list)
    t_per = defaultdict(list)
    for r in baseline_results:
        b_per[r['repo']].append(r['success'])
    for r in trained_results:
        t_per[r['repo']].append(r['success'])
    repos = sorted(set(b_per) | set(t_per))
    b_vals = [sum(b_per[r]) / len(b_per[r]) if b_per[r] else 0 for r in repos]
    t_vals = [sum(t_per[r]) / len(t_per[r]) if t_per[r] else 0 for r in repos]
    x = np.arange(len(repos))
    w = 0.35
    plt.figure(figsize=(8, 4))
    plt.bar(x - w/2, b_vals, w, label='Baseline', color='#bbbbbb')
    plt.bar(x + w/2, t_vals, w, label='Trained', color='#1f77b4')
    plt.xticks(x, repos, rotation=15)
    plt.ylabel('Pass rate')
    plt.legend()
    plt.title('Pass rate by repo')
    plt.tight_layout()
    plt.savefig('plots/per_repo_pass_rate.png', dpi=150, bbox_inches='tight')
    plt.close()

    b_turns = [r['turns'] for r in baseline_results if r['success']]
    t_turns = [r['turns'] for r in trained_results if r['success']]
    plt.figure(figsize=(7, 4))
    bins = list(range(1, 9))
    plt.hist([b_turns, t_turns], bins=bins, label=['Baseline', 'Trained'], color=['#bbbbbb', '#1f77b4'], edgecolor='white')
    plt.xlabel('Turns to completion')
    plt.ylabel('Count')
    plt.legend()
    plt.title('Completion turns distribution')
    plt.tight_layout()
    plt.savefig('plots/completion_turns.png', dpi=150, bbox_inches='tight')
    plt.close()

    print('OK: plots generated')
    print('Files:', sorted(os.listdir('plots')))